In [1]:
import sys

print(sys.executable)

c:\Users\ASUSTUF\Desktop\tubitak_internship\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("QMOF environment is ready!")

QMOF environment is ready!


In [1]:
from pathlib import Path

# Notebook notebooks/ klasöründe çalıştığı için
# bir üst klasör proje klasörümüzdür.
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Klasörler henüz yoksa otomatik oluştur.
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook çalışma klasörü:", Path.cwd())
print("Proje klasörü:", PROJECT_ROOT)
print("Ham veri klasörü:", RAW_DIR)
print("Ham veri klasörü mevcut mu?", RAW_DIR.exists())

Notebook çalışma klasörü: c:\Users\ASUSTUF\Desktop\tubitak_internship\notebooks
Proje klasörü: c:\Users\ASUSTUF\Desktop\tubitak_internship
Ham veri klasörü: c:\Users\ASUSTUF\Desktop\tubitak_internship\data\raw
Ham veri klasörü mevcut mu? True


In [2]:
all_files = [
    path
    for path in RAW_DIR.rglob("*")
    if path.is_file()
]

print(f"Toplam dosya sayısı: {len(all_files):,}")
print("\nİlk 30 dosya:\n")

for path in all_files[:30]:
    print(path.relative_to(RAW_DIR))

Toplam dosya sayısı: 8

İlk 30 dosya:

qmof_database\qmof.csv
qmof_database\qmof.json
qmof_database\qmof_structure_data.json
qmof_database\README.md
qmof_database\relaxed_structures.zip
qmof_database\unrelaxed_structures.zip
qmof_database\scripts\json_to_csv.py
qmof_database\scripts\make_cifs.py


In [3]:
from collections import Counter

file_type_counts = Counter(
    path.suffix.lower() if path.suffix else "[uzantısız]"
    for path in all_files
)

print("Dosya türleri:\n")

for extension, count in file_type_counts.most_common():
    print(f"{extension:15} {count:,}")

Dosya türleri:

.json           2
.zip            2
.py             2
.csv            1
.md             1


In [4]:
from zipfile import ZipFile
from collections import Counter

qmof_dir = RAW_DIR / "qmof_database"

structure_zip_paths = [
    qmof_dir / "relaxed_structures.zip",
    qmof_dir / "unrelaxed_structures.zip",
]

for zip_path in structure_zip_paths:
    print("=" * 70)
    print("Arşiv:", zip_path.name)

    if not zip_path.exists():
        print("Dosya bulunamadı.")
        continue

    with ZipFile(zip_path, "r") as zip_file:
        file_names = [
            name
            for name in zip_file.namelist()
            if not name.endswith("/")
        ]

    extension_counts = Counter(
        Path(name).suffix.lower() or "[uzantısız]"
        for name in file_names
    )

    print(f"Arşivdeki toplam dosya sayısı: {len(file_names):,}")
    print("Dosya türleri:", dict(extension_counts))

    print("\nİlk 10 dosya:")
    for name in file_names[:10]:
        print("-", name)

Arşiv: relaxed_structures.zip
Arşivdeki toplam dosya sayısı: 20,372
Dosya türleri: {'.cif': 20372}

İlk 10 dosya:
- relaxed_structures/qmof-0000295.cif
- relaxed_structures/qmof-00019ff.cif
- relaxed_structures/qmof-0001b0d.cif
- relaxed_structures/qmof-0003ae4.cif
- relaxed_structures/qmof-000512e.cif
- relaxed_structures/qmof-00052d0.cif
- relaxed_structures/qmof-0006638.cif
- relaxed_structures/qmof-000741d.cif
- relaxed_structures/qmof-00089fc.cif
- relaxed_structures/qmof-0009829.cif
Arşiv: unrelaxed_structures.zip
Arşivdeki toplam dosya sayısı: 4,161
Dosya türleri: {'.py': 1, '.gcd': 10, '.md': 2, '.txt': 1, '.cif': 4147}

İlk 10 dosya:
- unrelaxed_structures/csd/download_csd_cifs.py
- unrelaxed_structures/csd/gcd_files/intermediate_steps/CoRE_108.gcd
- unrelaxed_structures/csd/gcd_files/intermediate_steps/CoRE_14142.gcd
- unrelaxed_structures/csd/gcd_files/intermediate_steps/CoRE_2844.gcd
- unrelaxed_structures/csd/gcd_files/intermediate_steps/CoRE_3788.gcd
- unrelaxed_structure

# 1. QMOF Veri Setinin Yüklenmesi

Bu bölümde QMOF veri setinin tablo biçimindeki ana dosyası yüklenmektedir.

In [11]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
QMOF_DIR = PROJECT_ROOT / "data" / "raw" / "qmof_database"
CSV_PATH = QMOF_DIR / "qmof.csv"

print("CSV dosya yolu:", CSV_PATH)
print("Dosya mevcut mu?:", CSV_PATH.exists())

CSV dosya yolu: c:\Users\ASUSTUF\Desktop\tubitak_internship\data\raw\qmof_database\qmof.csv
Dosya mevcut mu?: True


In [12]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Veri seti başarıyla yüklendi.")
print("Satır sayısı:", df.shape[0])
print("Kolon sayısı:", df.shape[1])

Veri seti başarıyla yüklendi.
Satır sayısı: 20372
Kolon sayısı: 94


# 2. İlk Kayıtların İncelenmesi

Veri setinin ilk satırları incelenerek kolonların içerdiği değerler gözlemlenmektedir.

In [13]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

display(df.head())

,qmof_id,name,info.formula,info.formula_reduced,info.mofid.mofid,info.mofid.mofkey,info.mofid.smiles_nodes,info.mofid.smiles_linkers,info.mofid.smiles,info.mofid.topology,info.natoms,info.pld,info.lcd,info.density,info.volume,info.symmetry.spacegroup,info.symmetry.spacegroup_number,info.symmetry.spacegroup_crystal,info.symmetry.pointgroup,info.synthesized,info.source,info.doi,inputs.pbe.theory,inputs.pbe.pseudopotentials,inputs.pbe.encut,inputs.pbe.kpoints,inputs.pbe.gamma,inputs.pbe.spin,outputs.pbe.energy_total,outputs.pbe.energy_vdw,outputs.pbe.energy_elec,outputs.pbe.net_magmom,outputs.pbe.bandgap,outputs.pbe.cbm,outputs.pbe.vbm,outputs.pbe.directgap,outputs.pbe.bandgap_spins,outputs.pbe.cbm_spins,outputs.pbe.vbm_spins,outputs.pbe.directgap_spins,inputs.hle17.theory,inputs.hle17.pseudopotentials,inputs.hle17.encut,inputs.hle17.kpoints,inputs.hle17.gamma,inputs.hle17.spin,inputs.hse06_10hf.theory,inputs.hse06_10hf.pseudopotentials,inputs.hse06_10hf.encut,inputs.hse06_10hf.kpoints,inputs.hse06_10hf.gamma,inputs.hse06_10hf.spin,inputs.hse06.theory,inputs.hse06.pseudopotentials,inputs.hse06.encut,inputs.hse06.kpoints,inputs.hse06.gamma,inputs.hse06.spin,outputs.hle17.energy_total,outputs.hle17.energy_vdw,outputs.hle17.energy_elec,outputs.hle17.net_magmom,outputs.hle17.bandgap,outputs.hle17.cbm,outputs.hle17.vbm,outputs.hle17.directgap,outputs.hle17.bandgap_spins,outputs.hle17.cbm_spins,outputs.hle17.vbm_spins,outputs.hle17.directgap_spins,outputs.hse06_10hf.energy_total,outputs.hse06_10hf.energy_vdw,outputs.hse06_10hf.energy_elec,outputs.hse06_10hf.net_magmom,outputs.hse06_10hf.bandgap,outputs.hse06_10hf.cbm,outputs.hse06_10hf.vbm,outputs.hse06_10hf.directgap,outputs.hse06_10hf.bandgap_spins,outputs.hse06_10hf.cbm_spins,outputs.hse06_10hf.vbm_spins,outputs.hse06_10hf.directgap_spins,outputs.hse06.energy_total,outputs.hse06.energy_vdw,outputs.hse06.energy_elec,outputs.hse06.net_magmom,outputs.hse06.bandgap,outputs.hse06.cbm,outputs.hse06.vbm,outputs.hse06.directgap,outputs.hse06.bandgap_spins,outputs.hse06.cbm_spins,outputs.hse06.vbm_spins,outputs.hse06.directgap_spins
0,qmof-8a95c27,ABACUF01_FSR,Ba2CuC6H14O16,Ba2CuC6H14O16,NaN,NaN,"['O', '[Ba]', '[Cu]']",['[O-]C=O'],O.[Ba].[Cu].[O-]C=O,NaN,39,0.68822,1.35480,2.763246,408.857471,P-1,2,triclinic,-1,True,CSD,https://doi.org/10.1016/j.molstruc.2004.03.051,PBE-D3BJ,"['Ba_sv 06Sep2000', 'Cu 22Jun2005', 'H 15Jun2001', 'C 08Apr2002', 'O 08Apr2002']",520,"[3, 3, 2]",True,True,-238.661417,-3.38941,-235.272007,1.0,0.632527,1.237645,0.605118,False,"[3.678962, 0.6325270000000001]","[3.982388, 1.237645]","[0.303426, 0.605118]","[False, False]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,qmof-019ba28,ABALOF_FSR,Cu12C36H56I16N4S4,Cu3C9H14I4NS,NaN,NaN,NaN,NaN,NaN,NaN,128,1.18570,2.13507,3.229952,1781.965032,P2_1/c,14,monoclinic,2/m,True,CSD,https://doi.org/10.1021/ja048624i,PBE-D3BJ,"['Cu 22Jun2005', 'H 15Jun2001', 'C 08Apr2002', 'S 06Sep2000', 'I 08Apr2002', 'N 08Apr2002']",520,"[2, 2, 1]",True,False,-672.046744,-15.98125,-656.065494,0.0,1.134232,3.430440,2.296208,False,"[None, None]","[None, None]","[None, None]","[None, None]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,qmof-830ed1c,ABAVIJ_FSR,Co4C48H32N8O16,CoC12H8N2O4,[Co].[O-]C(=O)c1ccncc1 MOFid-v1.rtl.cat0,Co.TWBYWOBDOCUKOW.MOFkey-v1.rtl,['[Co]'],['[O-]C(=O)c1ccncc1'],[Co].[O-]C(=O)c1ccncc1,rtl,108,2.36128,4.21176,1.557644,1292.643180,Cc,9,monoclinic,m,True,CSD,https://doi.org/10.1039/b404485a,PBE-D3BJ,"['Co 02Aug2007', 'H 15Jun2001', 'C 08Apr2002', 'N 08Apr2002', 'O 08Apr2002']",520,"[2, 2, 1]",True,True,-759.996078,-8.60846,-751.387618,12.0,0.345448,1.091140,0.745692,False,"[1.5779139999999998

# 3. Kolon İsimlerinin İncelenmesi

In [14]:
print(f"Toplam kolon sayısı: {len(df.columns)}\n")

for index, column in enumerate(df.columns, start=1):
    print(f"{index:3}. {column}")

Toplam kolon sayısı: 94

  1. qmof_id
  2. name
  3. info.formula
  4. info.formula_reduced
  5. info.mofid.mofid
  6. info.mofid.mofkey
  7. info.mofid.smiles_nodes
  8. info.mofid.smiles_linkers
  9. info.mofid.smiles
 10. info.mofid.topology
 11. info.natoms
 12. info.pld
 13. info.lcd
 14. info.density
 15. info.volume
 16. info.symmetry.spacegroup
 17. info.symmetry.spacegroup_number
 18. info.symmetry.spacegroup_crystal
 19. info.symmetry.pointgroup
 20. info.synthesized
 21. info.source
 22. info.doi
 23. inputs.pbe.theory
 24. inputs.pbe.pseudopotentials
 25. inputs.pbe.encut
 26. inputs.pbe.kpoints
 27. inputs.pbe.gamma
 28. inputs.pbe.spin
 29. outputs.pbe.energy_total
 30. outputs.pbe.energy_vdw
 31. outputs.pbe.energy_elec
 32. outputs.pbe.net_magmom
 33. outputs.pbe.bandgap
 34. outputs.pbe.cbm
 35. outputs.pbe.vbm
 36. outputs.pbe.directgap
 37. outputs.pbe.bandgap_spins
 38. outputs.pbe.cbm_spins
 39. outputs.pbe.vbm_spins
 40. outputs.pbe.directgap_spins
 41. inputs.hle

# 4. Veri Tipleri ve Dolu Kayıt Sayıları

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20372 entries, 0 to 20371
Data columns (total 94 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   qmof_id                             20372 non-null  object 
 1   name                                20372 non-null  object 
 2   info.formula                        20372 non-null  object 
 3   info.formula_reduced                20372 non-null  object 
 4   info.mofid.mofid                    7462 non-null   object 
 5   info.mofid.mofkey                   7695 non-null   object 
 6   info.mofid.smiles_nodes             17677 non-null  object 
 7   info.mofid.smiles_linkers           17544 non-null  object 
 8   info.mofid.smiles                   17677 non-null  object 
 9   info.mofid.topology                 7900 non-null   object 
 10  info.natoms                         20372 non-null  int64  
 11  info.pld                            20372

# 5. Eksik Değer Analizi

Her kolondaki eksik değer sayısı ve oranı incelenmektedir.

In [16]:
null_summary = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_percentage": df.isna().mean() * 100
})

null_summary = null_summary.sort_values(
    by="null_count",
    ascending=False
)

display(null_summary)

,null_count,null_percentage
info.mofid.mofid,12910,63.371294
info.mofid.mofkey,12677,62.227567
info.mofid.topology,12472,61.221284
outputs.hle17.cbm_spins,9616,47.202042
outputs.hle17.energy_elec,9616,47.202042
...,...,...
outputs.pbe.vbm,0,0.000000
outputs.pbe.cbm,0,0.000000
outputs.pbe.bandgap,0,0.000000
outputs.pbe.energy_elec,0,0.000000


# 6. Tekrarlanan Kayıtların Kontrolü

In [17]:
duplicate_count = df.duplicated().sum()

print("Tamamen aynı olan tekrarlı satır sayısı:", duplicate_count)

Tamamen aynı olan tekrarlı satır sayısı: 0


# 7. Sayısal Kolonların İncelenmesi

In [18]:
numeric_df = df.select_dtypes(include="number")

print("Sayısal kolon sayısı:", numeric_df.shape[1])

for column in numeric_df.columns:
    print(column)

Sayısal kolon sayısı: 38
info.natoms
info.pld
info.lcd
info.density
info.volume
info.symmetry.spacegroup_number
inputs.pbe.encut
outputs.pbe.energy_total
outputs.pbe.energy_vdw
outputs.pbe.energy_elec
outputs.pbe.net_magmom
outputs.pbe.bandgap
outputs.pbe.cbm
outputs.pbe.vbm
inputs.hle17.encut
inputs.hse06_10hf.encut
inputs.hse06.encut
outputs.hle17.energy_total
outputs.hle17.energy_vdw
outputs.hle17.energy_elec
outputs.hle17.net_magmom
outputs.hle17.bandgap
outputs.hle17.cbm
outputs.hle17.vbm
outputs.hse06_10hf.energy_total
outputs.hse06_10hf.energy_vdw
outputs.hse06_10hf.energy_elec
outputs.hse06_10hf.net_magmom
outputs.hse06_10hf.bandgap
outputs.hse06_10hf.cbm
outputs.hse06_10hf.vbm
outputs.hse06.energy_total
outputs.hse06.energy_vdw
outputs.hse06.energy_elec
outputs.hse06.net_magmom
outputs.hse06.bandgap
outputs.hse06.cbm
outputs.hse06.vbm


In [19]:
numeric_statistics = numeric_df.describe().T

numeric_statistics["median"] = numeric_df.median()
numeric_statistics["null_count"] = numeric_df.isna().sum()

display(numeric_statistics)

,count,mean,std,min,25%,50%,75%,max,median,null_count
info.natoms,20372.0,113.525574,68.780485,17.000000,68.000000,96.000000,136.000000,500.000000,96.000000,0
info.pld,20372.0,2.958458,3.524561,0.000000,1.005290,1.327005,3.499610,44.416710,1.327005,0
info.lcd,20372.0,4.459336,4.110913,0.783710,2.002782,2.716785,5.068950,44.893720,2.716785,0
info.density,20372.0,1.695565,0.646327,0.087070,1.335668,1.698563,2.063290,5.436020,1.698563,0
info.volume,20372.0,1878.164019,2365.106540,170.866767,791.240916,1140.287904,2063.258440,49206.400520,1140.287904,0
info.symmetry.spacegroup_number,20372.0,19.133664,36.781876,1.000000,2.000000,9.000000,15.000000,230.000000,9.000000,0
inputs.pbe.encut,20372.0,520.000000,0.000000,520.000000,520.000000,520.000000,520.000000,520.000000,520.000000,0
outputs.pbe.energy_total,20372.0,-758.047747,468.137309,-3449.382161,-908.087510,-634.498316,-452.002111,-110.014150,-634.498316,0
outputs.pbe.energy_vdw,20372.0,-8.708817,4.803020,-47.896540,-10.373985,-7.673480,-5.657668,-1.478460,-7.673480,0
outputs.pbe.energy_elec,20372.0,-749.338930,463.683262,-3413.768900,-897.806123,-626.515762,-445.807212,-108.019150,-626.515762,0


# 8. Veri Seti Açıklamalarının İncelenmesi

In [20]:
README_PATH = QMOF_DIR / "README.md"

readme_text = README_PATH.read_text(encoding="utf-8")

print(readme_text[:10000])

# Notes

## MOF Explorer

See http://materialsproject.org/mofs for an interactive version of the QMOF Database along with additional documentation, which includes background information for how the calculations were run and information on how to properly cite this dataset.

## Pre-requisites

The instructions for handling the JSON files below assume you have Python installed. If you don't have Python, we recommend installing it via [Anaconda](https://www.anaconda.com). Pymatgen must also be installed, which can be done via `pip install pymatgen` in the command line.

# File Descriptions

## Tabulated Properties

The most pertinent properties for each structure are stored in two JSON files:

- `qmof.json`: Tabulated properties that are for the entire material, such as energies and band gaps, as well as pertinent information about the material.
- `qmof_structure_data.json`: Pymatgen structure objects containing the PBE-D3(BJ) relaxed structures, which are annotated with site-specific dat

### Eksik Değerler Hakkında İlk Gözlem

PBE hesaplama sonuçlarının 20.372 yapının tamamında mevcut olduğu
gözlenmiştir. Buna karşılık HLE17, HSE06-10HF ve HSE06 sonuçları
yalnızca yaklaşık 10.800 yapı için bulunmaktadır.

Bu kolonlardaki eksik değerler veri kaybından değil, ilgili
kuantum-kimyasal hesaplamanın her yapı için gerçekleştirilmemiş
olmasından kaynaklanmaktadır. Bu nedenle bu değerlerin ortalama veya
medyan ile doldurulması uygun değildir. Model hedefi olarak bu
kolonlardan biri seçilirse yalnızca hedef değeri mevcut olan kayıtlar
kullanılmalıdır.